# GR-TFiLM diffssl LSTM — 02b mirror + GR curve as temporal-FiLM conditioning

**Google Colab**: Runtime → **GPU**. Open via *File → Open notebook → GitHub*
(`5aola/Virtual-Analogue-Compressor-Modelling`); cell 1 clones the repo for the
`06_output` modules and mounts Drive for the dataset. **Push local changes before running.**

## Idea — the 02b diffssl LSTM, with the GR curve added via a true TFiLM

This is a faithful mirror of `02b_sota_training/train_lstm_diffssl_tvc.ipynb`
(the diffssl `LSTM32TVC` recipe) with **one addition**: the exported gain-reduction
curve modulates the LSTM hidden features through a **temporal FiLM** (γ/β), the
mechanism from the nablafx fork (`modules.py::TFiLM`) — *not* the concat-style
`tvcond`.

```
raw dry ─┐                         4 static knobs
         │  tvcond (TVFiLMCond, UNCHANGED from 02b): pool(|x|)⊕knobs → cond_seq[16]
         └── cat(x, cond_seq) → main LSTM(17→32) → hidden
                                        │
              GR curve ─► GR-TFiLM: pool(GR) → blockLSTM → γ,β → modulate hidden
                                        │
                                   Linear(32→1) → tanh → wet
```

- **Static knobs** (threshold/attack/release/ratio): conditioned exactly like the
  SOTA — `TVFiLMCond` pools `|x|`, concatenates the 4 knobs, and the block-rate LSTM
  emits a `cond_seq` that is concatenated to the main LSTM input (`LSTM(17→32)`).
- **GR curve** (`GRTFiLMDiffSSLLSTM.gr_tfilm`): a temporal FiLM whose block-rate LSTM
  reads the (reduction-positive, peak-pooled) GR and produces per-block γ/β that
  modulate the 32-dim hidden features. This is the privileged, ground-truth envelope
  that diffssl had to *learn* from `|x|`, injected here as a proper FiLM modulation.

## Identical to 02b
- **Model core**: diffssl `LSTM32TVC` (`cond_type="tvcond"`, raw input, direct output + `tanh`).
- **Dataset / split**: Diff-SSL-G-Comp, 10 settings × 10 songs, seed 42, **v2 external
  `test_ground_truth` policy** — the held-out test set is exactly the
  `test_ground_truth/` pairs (excluded from train/val by key); val = 1 remaining
  song × all settings. **Shared with 02b and the coloration black-box**, so every
  model in the ablation scores on the same test set.
- **Training recipe**: 3 s crops, `batch_size=64`, LR `2e-3`, state reset every batch,
  TBPTT sub-steps of 4410; `0.5·L1 + 0.5·MR-STFT`, AdamW, fixed **100-epoch** budget.

**Recipe: cosine + bf16 (matches the other `06_output` models).** This run uses a
**cosine LR** schedule over the 100 epochs (`SCHEDULER="cosine"`) and **bf16 autocast**
around the LSTM forward (`USE_AMP`) — the exact recipe of
`train_lstm_color_blackbox.ipynb`, so the GR-TFiLM row is directly comparable to the
other 06_output models. For exact parity with the *02b baseline* (which uses
`ReduceLROnPlateau` + fp32), set `SCHEDULER="plateau"`, `USE_AMP=False`. Note also
that `loss/train` is logged as the mean of the per-sub-step losses (val/test report
the exact full-crop loss); only the train-loss *curve* is affected.

The GR-TFiLM adds parameters (the temporal-FiLM LSTM emits `2·hidden_size` for γ/β),
so this model (~25k params) is larger than the ~8k-param SOTA baseline — the capacity
confound to keep in mind when reading the ablation.

In [1]:
# -- 0. Dependencies ---------------------------------------------------
# This variant uses nablafx (TVFiLMMod). Pin numpy first so lightning/nablafx
# installs can't downgrade Colab's numpy 2.x and break torch. Install
# lightning/nablafx --no-deps so they can't clobber Colab's CUDA torch.
# `rational` / `frechet_audio_distance` are nablafx import-chain deps we never
# use here; stub both so `from nablafx...` doesn't drag in broken wheels.
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile auraloss einops lightning-utilities packaging
!pip install -q --no-deps lightning nablafx

import sys, types

rational = types.ModuleType("rational")
rational.torch = types.ModuleType("rational.torch")
rational.torch.Rational = type("Rational", (), {})
sys.modules["rational"], sys.modules["rational.torch"] = rational, rational.torch

fad = types.ModuleType("frechet_audio_distance")
fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
sys.modules["frechet_audio_distance"] = fad

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} - restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 64.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.8/249.8 kB 28.1 MB/s eta 0:00:00
numpy 2.0.2, torch 2.11.0+cu128


In [2]:
# -- 1. Mount Drive (dataset) + clone repo from GitHub (code) ---------
# The repo is NOT synced to Drive (only data/ is). Code comes from GitHub -
# push local changes before (re)running this cell; re-running pulls updates.

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" reset --hard origin/main
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT

# Module directory for this notebook (dataset_tfilm/model_tfilm/system_tfilm live here).
COND_DIR = os.path.join(REPO_ROOT, "06_output")
assert os.path.isfile(os.path.join(COND_DIR, "dataset_tfilm.py")), (
    f"Clone failed or stale: {COND_DIR}. Did you push local changes?"
)

OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "diffssl_gr_tfilm_runs")

assert os.path.isdir(os.path.join(DATA_ROOT, "gr_curves")), f"Bad DATA_ROOT: {DATA_ROOT}"
assert os.path.isdir(os.path.join(DATA_ROOT, "processed_ground_truth")), "Missing wet audio dir"
assert os.path.isdir(os.path.join(DATA_ROOT, "test_ground_truth")), (
    f"No test_ground_truth/ under {DATA_ROOT} — it defines the held-out test set "
    "(v2 split policy, shared with 02b); sync it to Drive before training."
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Drop cached local modules so a prior run cannot keep stale classes.
for _name in list(sys.modules):
    if _name in ("dataset_tfilm", "model_tfilm", "system_tfilm", "splits", "amplitude_match"):
        del sys.modules[_name]

# repo root (for `src` + `nablafx`) + module dir (for dataset_tfilm/model_tfilm/...)
for p in (REPO_ROOT, os.path.join(REPO_ROOT, "nablafx"), COND_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"COND_DIR   : {COND_DIR}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

Mounted at /content/drive
Cloning into '/content/Virtual-Analogue-Compressor-Modelling'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (257/257), done.
remote: Compressing objects: 100% (240/240), done.
remote: Total 257 (delta 19), reused 166 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (257/257), 132.08 MiB | 18.41 MiB/s, done.
Resolving deltas: 100% (19/19), done.
Updating files: 100% (229/229), done.
REPO_ROOT  : /content/Virtual-Analogue-Compressor-Modelling
COND_DIR   : /content/Virtual-Analogue-Compressor-Modelling/06_output
DATA_ROOT  : /content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp
OUTPUT_DIR : /content/drive/Othercomputers/MacBook Air/data/diffssl_gr_tfilm_runs


In [3]:
# -- 2. Cache dataset to Colab local SSD ------------------------------
# The GR-TFiLM model needs dry (input) + GR curves (conditioning) + wet (target).
# Wet/gr are mirrored under their ORIGINAL subfolders (processed_ground_truth vs
# test_ground_truth) so the external test pairs stay under test_ground_truth/
# on the cache too (the v2 split treats that folder as test-only).

import shutil
from dataset_tfilm import discover_gr_pairs

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"

pairs = discover_gr_pairs(DATA_ROOT, include_test_ground_truth=True)
settings = sorted({p["setting"] for p in pairs})
songs = sorted({p["song"] for p in pairs})
print(f"Caching {len(songs)} songs x {len(settings)} settings ({len(pairs)} pairs) -> {LOCAL_DATA_ROOT}")

def _mirror(src, dst):
    src, dst = Path(src), Path(dst)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

# dry WAVs (one per song, shared across settings)
for song in songs:
    fn = f"{song}_UnmasteredWAV.wav"
    _mirror(Path(DATA_ROOT) / "processed_normalized" / fn,
            Path(LOCAL_DATA_ROOT) / "processed_normalized" / fn)

# GR curves (.pt) + wet WAVs (-exported.wav), per pair, preserving the source
# subfolder (processed_ground_truth vs test_ground_truth)
for p in pairs:
    _mirror(p["gr"], Path(LOCAL_DATA_ROOT) / Path(p["gr"]).relative_to(DATA_ROOT))
    _mirror(p["wet"], Path(LOCAL_DATA_ROOT) / Path(p["wet"]).relative_to(DATA_ROOT))

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")

Caching 15 songs x 15 settings (105 pairs) -> /content/Diff-SSL-G-Comp
Using local cache: /content/Diff-SSL-G-Comp


In [4]:
# -- 3. Imports & hyper-parameters (02b LSTM32TVC + GR-TFiLM) ---------

import importlib
import json
from datetime import datetime

import torch
import lightning as pl
from lightning.pytorch.callbacks import (
    LearningRateMonitor, ModelCheckpoint, TQDMProgressBar,
)
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

import dataset_tfilm as _dataset_tfilm
importlib.reload(_dataset_tfilm)
from dataset_tfilm import (
    SAMPLE_LENGTH, SAMPLE_RATE, GRCropDataModule, discover_gr_pairs,
)  # BATCH_SIZE set explicitly below (02b L4 tuning), not imported

import model_tfilm as _model_tfilm
importlib.reload(_model_tfilm)
from model_tfilm import GRTFiLMDiffSSLLSTM

import system_tfilm as _system_tfilm
importlib.reload(_system_tfilm)
from system_tfilm import GRTFiLMSystem

from splits import (
    DIFFSSL_PARAM_RANGES, build_split_manifest, discover_test_ground_truth_keys,
)
from src.dsp import PARAM_ORDER

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

# -- split: v2 external test_ground_truth policy (shared with 02b / the other
#    06_output notebooks) — the ONLY held-out test set is test_ground_truth/,
#    excluded from train/val by key; every other pair trains or validates. --
SPLIT_SEED   = 42
N_VAL_SONGS  = 1

# -- training: the 02b LSTM32TVC recipe (diffssl BlackBoxSystemWithTBPTT: TBPTT
#    sub-steps, state reset every batch) at 02b's L4-speed tuning (batch 64 /
#    LR 2e-3), with a cosine LR schedule + bf16 autocast — the SAME recipe as
#    the coloration black-box (train_lstm_color_blackbox.ipynb), so the GR-TFiLM
#    row is comparable to the other 06_output models. Flip SCHEDULER="plateau" /
#    USE_AMP=False for exact 02b-baseline parity (plateau + fp32). --
BATCH_SIZE       = 64       # diffssl default is 16; 64 far better utilises the L4
LR               = 2e-3     # sqrt-scaled: 1e-3 * sqrt(64/16)
MAX_EPOCHS       = 100      # fixed budget == cosine T_max
STEP_NUM_SAMPLES = 4410     # diffssl TBPTT sub-step (0.1 s). Raise (e.g. 22050)
                            #   for fewer optimizer steps / faster epochs.
SCHEDULER        = "cosine"  # "cosine" (per-epoch, fits the fixed budget) | "plateau" | "none"
ETA_MIN          = 1e-6      # cosine floor
USE_AMP          = True      # bf16 autocast around the LSTM forward (cuda only)
CHECK_VAL_EVERY_N_EPOCH = 1  # raise to 2-5 to spend less time in validation

# -- model core (diffssl LSTM32TVC: tvcond on |x| + 4 static knobs) --
HIDDEN_SIZE     = 32        # LSTM32TVC
NUM_LAYERS      = 1
NUM_CONTROLS    = 4
TVCOND_DIM      = 16        # diffssl cond_dim (fixed at 16)
COND_BLOCK_SIZE = 128       # diffssl tvcond block (128/44100 ~= 2.9 ms)
COND_NUM_LAYERS = 1

# -- GR conditioning (temporal FiLM: pool(GR) -> blockLSTM -> gamma,beta) --
GR_TFILM_BLOCK_SIZE = 128
GR_TFILM_NUM_LAYERS = 1

# -- loss (SOTA waveform recipe, handled inside GRTFiLMSystem) --
#    0.5*L1 + 0.5*MR-STFT  (metrics: esr / rmse / mae / mse)

RUN_TAG    = "diffssl_lstm32_tvc_gr_tfilm"
# The old resume run was trained under the legacy split (lowest-threshold test)
# + the pre-migration recipe; its split manifest + optimizer state are
# incompatible with the v2 external-test / cosine recipe, so start a fresh run.
RESUME_RUN = None

NVIDIA L4


In [5]:
# -- 4. Preview split — v2 external test_ground_truth policy ----------
# The ONLY held-out test set is test_ground_truth/ (scanned on DRIVE; the local
# cache mirrors it under the same subfolder). Its pairs are excluded from
# train/val BY KEY; every remaining (song, setting) pair trains or validates.

TEST_KEYS = discover_test_ground_truth_keys(DRIVE_DATA_ROOT)
print(f"test_ground_truth pairs ({len(TEST_KEYS)}):")
for k in sorted(TEST_KEYS):
    print(f"  {k}")

preview = build_split_manifest(
    discover_gr_pairs(DATA_ROOT, include_test_ground_truth=True),
    seed=SPLIT_SEED, n_val_songs=N_VAL_SONGS, test_pair_keys=TEST_KEYS,
)
leaked = (set(preview.train_pair_keys) | set(preview.val_pair_keys)) & TEST_KEYS
assert not leaked, f"test_ground_truth pairs leaked into train/val: {sorted(leaked)}"

print(f"\nSettings ({len(preview.all_settings)}): {preview.all_settings}")
print(f"Train songs: {preview.train_songs}")
print(f"Val songs  : {preview.val_songs}")
print(f"Test pairs : {preview.test_pair_keys}")
print(f"Pairs - train={len(preview.train_pair_keys)} "
      f"val={len(preview.val_pair_keys)} test={len(preview.test_pair_keys)}")

test_ground_truth pairs (5):
  54::threshold_12_attack_1_release_0.1_ratio_2
  Convertible::threshold_4_attack_30_release_0.8_ratio_10
  IncidenteEnIntag::threshold_-12_attack_1_release_0.8_ratio_2
  OralHygiene::threshold_-8_attack_3_release_0.4_ratio_4
  SuchFinePeople::threshold_4_attack_1_release_0.8_ratio_10

Settings (15): ['threshold_-12_attack_10_release_0.4_ratio_10', 'threshold_-12_attack_1_release_0.1_ratio_2', 'threshold_-12_attack_1_release_0.8_ratio_2', 'threshold_-4_attack_10_release_0.1_ratio_2', 'threshold_-4_attack_1_release_0.4_ratio_10', 'threshold_-8_attack_30_release_0.8_ratio_4', 'threshold_-8_attack_3_release_0.4_ratio_4', 'threshold_0_attack_3_release_0.8_ratio_4', 'threshold_12_attack_1_release_0.1_ratio_2', 'threshold_12_attack_3_release_0.8_ratio_2', 'threshold_4_attack_10_release_0.1_ratio_10', 'threshold_4_attack_1_release_0.8_ratio_10', 'threshold_4_attack_30_release_0.8_ratio_10', 'threshold_8_attack_1_release_0.1_ratio_10', 'threshold_8_attack_30_releas

In [6]:
# -- 5. Model size ----------------------------------------------------

model = GRTFiLMDiffSSLLSTM(
    num_controls=NUM_CONTROLS, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS,
    tvcond_dim=TVCOND_DIM, cond_block_size=COND_BLOCK_SIZE, cond_num_layers=COND_NUM_LAYERS,
    gr_tfilm_block_size=GR_TFILM_BLOCK_SIZE, gr_tfilm_num_layers=GR_TFILM_NUM_LAYERS,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"GRTFiLMDiffSSLLSTM: {n_params:,} params  "
      f"(hidden={HIDDEN_SIZE}, tvcond_dim={TVCOND_DIM}, controls={NUM_CONTROLS})")
for name, mod in model.named_children():
    print(f"  {name:10s} {sum(p.numel() for p in mod.parameters()):,}")
print(f"\ncond_nn = TVFiLMCond (SOTA tvcond on |x| + {NUM_CONTROLS} knobs) | "
      f"gr_tfilm = temporal-FiLM on GR")
print(f"Crop {SAMPLE_LENGTH} ({SAMPLE_LENGTH/SAMPLE_RATE:.2f}s) | {SAMPLE_RATE} Hz | "
      f"TBPTT step {STEP_NUM_SAMPLES} | tvcond block {COND_BLOCK_SIZE} "
      f"({COND_BLOCK_SIZE/SAMPLE_RATE*1e3:.1f} ms) | GR-TFiLM block {GR_TFILM_BLOCK_SIZE}")


GRTFiLMDiffSSLLSTM: 25,185 params  (hidden=32, tvcond_dim=16, controls=4)
  cond_nn    1,472
  lstm       6,528
  gr_tfilm   17,152
  lin        33

cond_nn = TVFiLMCond (SOTA tvcond on |x| + 4 knobs) | gr_tfilm = temporal-FiLM on GR
Crop 132300 (3.00s) | 44100 Hz | TBPTT step 4410 | tvcond block 128 (2.9 ms) | GR-TFiLM block 128


In [7]:
# -- 6. Train ---------------------------------------------------------

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

# Multi-worker loading: the model is small, so without this the GPU starves on
# the per-item soundfile seeks (dry + wet). Cap at 8 (matches 02b).
NUM_WORKERS = min(8, os.cpu_count() or 2)
print(f"DataLoader num_workers: {NUM_WORKERS}")

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"gr_tfilm_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = GRCropDataModule(
    data_root=DATA_ROOT, sample_length=SAMPLE_LENGTH, sample_rate=SAMPLE_RATE,
    batch_size=BATCH_SIZE, split_seed=SPLIT_SEED, n_val_songs=N_VAL_SONGS,
    test_gt_root=DRIVE_DATA_ROOT,   # test keys scanned on Drive (source of truth)
    split_manifest_path=split_path, num_workers=NUM_WORKERS,
)
dm.setup()
print(f"Train/val/test crops: {len(dm.train_dataset)} / {len(dm.val_dataset)} / {len(dm.test_dataset)}")
print(f"Batches/epoch (train): {len(dm.train_dataloader())}  (batch_size={BATCH_SIZE})")

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "diffssl_lstm32_tvc + gr_temporal_film",
        "model_type": "GRTFiLMDiffSSLLSTM",
        "model_ref": "02b LSTM32TVC (tvcond) + nablafx-fork TFiLM on the GR curve",
        "dataset": "Diff-SSL-G-Comp", "setting": "multi (all non-test settings, tvcond on 4 knobs)",
        "conditioning": "knobs via tvcond (TVFiLMCond); GR via temporal FiLM (gamma/beta)",
        "sample_rate": SAMPLE_RATE, "sample_length": SAMPLE_LENGTH, "batch_size": BATCH_SIZE,
        "step_num_samples": STEP_NUM_SAMPLES,
        "param_order": PARAM_ORDER, "param_ranges": DIFFSSL_PARAM_RANGES,
        "split_seed": SPLIT_SEED,
        "split_policy": "external_test_ground_truth (test pairs excluded from train/val by key)",
        "test_pair_keys": sorted(TEST_KEYS),
        "train_songs": dm.split.train_songs,
        "val_songs": dm.split.val_songs, "test_songs": dm.split.test_songs,
        "test_settings": dm.split.test_settings,
        "model": {"hidden_size": HIDDEN_SIZE, "num_layers": NUM_LAYERS,
                   "num_controls": NUM_CONTROLS, "tvcond_dim": TVCOND_DIM,
                   "cond_block_size": COND_BLOCK_SIZE, "cond_num_layers": COND_NUM_LAYERS,
                   "gr_tfilm_block_size": GR_TFILM_BLOCK_SIZE,
                   "gr_tfilm_num_layers": GR_TFILM_NUM_LAYERS, "num_params": n_params},
        "loss": "0.5*L1 + 0.5*MR-STFT", "metrics": ["esr", "rmse", "mae", "mse"],
        "optimizer": f"adamw + {SCHEDULER}",
        "scheduler": SCHEDULER, "eta_min": ETA_MIN, "use_amp": USE_AMP,
        "check_val_every_n_epoch": CHECK_VAL_EVERY_N_EPOCH,
        "training": "diffssl_crop_batches + tbptt_substeps (reset each batch)",
        "lr": LR, "max_epochs": MAX_EPOCHS,
    }, f, indent=2)

system = GRTFiLMSystem(
    model=model, lr=LR, step_num_samples=STEP_NUM_SAMPLES,
    scheduler=SCHEDULER, max_epochs=MAX_EPOCHS, eta_min=ETA_MIN, use_amp=USE_AMP,
)

ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(dirpath=ckpt_dir, monitor="loss/val", mode="min", save_top_k=3,
                    save_last=True, filename="best-{epoch:03d}-{step}",
                    auto_insert_metric_name=False),
    LearningRateMonitor(logging_interval="epoch"),
    TQDMProgressBar(refresh_rate=10),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator="gpu", devices=1,
    callbacks=callbacks, logger=loggers, log_every_n_steps=10,
    check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
)
trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"Best val loss: {callbacks[0].best_model_score:.6f}")
print(f"Best ckpt    : {callbacks[0].best_model_path}")

DataLoader num_workers: 8
NEW run: gr_tfilm_20260708_100749_diffssl_lstm32_tvc_gr_tfilm
Split seed     : 42
Train songs    : ['Air', 'BackroomInTulsa', 'Borderline', 'Ecstasy', 'Electrvm', 'LivingLie', 'NosPalpitants', 'OpenFire', 'SongForJohn']
Val songs      : ['AncoraQui']
Test pairs     : ['54::threshold_12_attack_1_release_0.1_ratio_2', 'Convertible::threshold_4_attack_30_release_0.8_ratio_10', 'IncidenteEnIntag::threshold_-12_attack_1_release_0.8_ratio_2', 'OralHygiene::threshold_-8_attack_3_release_0.4_ratio_4', 'SuchFinePeople::threshold_4_attack_1_release_0.8_ratio_10'] (external test_ground_truth)
Pair counts    : train=90 val=10 test=5 / 105 total
GRCropDataset: 8000 crops from 90 pairs  [sample_length=132300, 400.0 min audio]
GRCropDataset: 770 crops from 10 pairs  [sample_length=132300, 38.5 min audio]


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


GRCropDataset: 351 crops from 5 pairs  [sample_length=132300, 17.6 min audio]
Train/val/test crops: 8000 / 770 / 351
Batches/epoch (train): 125  (batch_size=64)


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name   ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model  │ GRTFiLMDiffSSLLSTM      │ 25.2 K │ train │     0 │
│ 1 │ l1     │ L1Loss                  │      0 │ train │     0 │
│ 2 │ mrstft │ MultiResolutionSTFTLoss │      0 │ train │     0 │
└───┴────────┴─────────────────────────┴────────┴───────┴───────┘

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=100` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.


Best val loss: 0.093279
Best ckpt    : /content/drive/Othercomputers/MacBook Air/data/diffssl_gr_tfilm_runs/gr_tfilm_20260708_100749_diffssl_lstm32_tvc_gr_tfilm/checkpoints/best-089-337500.ckpt


In [8]:
# -- 7. Test (external test_ground_truth pairs — unseen songs & settings) --

best_ckpt = callbacks[0].best_model_path or os.path.join(ckpt_dir, "last.ckpt")
print(f"Testing with: {best_ckpt}")
trainer.test(system, datamodule=dm, ckpt_path=best_ckpt)

INFO: Restoring states from the checkpoint path at /content/drive/Othercomputers/MacBook Air/data/diffssl_gr_tfilm_runs/gr_tfilm_20260708_100749_diffssl_lstm32_tvc_gr_tfilm/checkpoints/best-089-337500.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/drive/Othercomputers/MacBook Air/data/diffssl_gr_tfilm_runs/gr_tfilm_20260708_100749_diffssl_lstm32_tvc_gr_tfilm/checkpoints/best-089-337500.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loaded model weights from the checkpoint at /content/drive/Othercomputers/MacBook Air/data/diffssl_gr_tfilm_runs/gr_tfilm_20260708_100749_diffssl_lstm32_tvc_gr_tfilm/checkpoints/best-089-337500.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Loaded model weights from the checkpoint at /content/drive/Othercomputers/MacBook Air/data/diffssl_gr_tfilm_runs/gr_tfilm_20260708_100749_diffssl_lstm32_tvc_gr_tfilm/checkp

Testing with: /content/drive/Othercomputers/MacBook Air/data/diffssl_gr_tfilm_runs/gr_tfilm_20260708_100749_diffssl_lstm32_tvc_gr_tfilm/checkpoints/best-089-337500.ckpt


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         esr/test          │    2.5415608882904053     │
│         loss/test         │    0.12681593000888824    │
│       loss/test_fd        │    0.23253963887691498    │
│       loss/test_td        │   0.021092195063829422    │
│         mae/test          │   0.021092195063829422    │
│         mse/test          │   0.0009900227887555957   │
│         rmse/test         │   0.0007220834377221763   │
└───────────────────────────┴───────────────────────────┘

[{'loss/test': 0.12681593000888824,
  'loss/test_td': 0.021092195063829422,
  'loss/test_fd': 0.23253963887691498,
  'mae/test': 0.021092195063829422,
  'mse/test': 0.0009900227887555957,
  'esr/test': 2.5415608882904053,
  'rmse/test': 0.0007220834377221763}]

In [9]:
# -- 8. Plot: dry input vs prediction vs target -----------------------
# Crops reset state every batch (diffssl regime), so just reset_states() before
# each batch and run the model. Plots a few examples from one val batch.

import matplotlib.pyplot as plt
import numpy as np
from system_tfilm import esr_metric

best = torch.load(callbacks[0].best_model_path, map_location="cuda", weights_only=False)
system.load_state_dict(best["state_dict"])
system.eval().cuda()
print(f"Loaded best checkpoint: {callbacks[0].best_model_path}")

val_batches = list(dm.val_dataloader())
dry, gr, wet, params = val_batches[len(val_batches) // 2]

with torch.no_grad():
    system.model.reset_states()
    pred = system.model(dry.cuda(), gr.cuda(), params.cuda()).cpu()

dry_np, wet_np, pred_np = dry.numpy(), wet.numpy(), pred.numpy()
n_plots = min(4, dry_np.shape[0])
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 3 * n_plots), sharex=True, squeeze=False)
t = np.arange(wet_np.shape[-1]) / SAMPLE_RATE
for ax, r in zip(axes[:, 0], range(n_plots)):
    ax.plot(t, dry_np[r, 0], label="Dry (input)", alpha=0.4, lw=0.5, color="gray")
    ax.plot(t, wet_np[r, 0], label="Target (wet)", alpha=0.8, lw=0.5)
    ax.plot(t, pred_np[r, 0], label="Predicted", alpha=0.8, lw=0.5)
    pv = torch.from_numpy(pred_np[r]); tv = torch.from_numpy(wet_np[r])
    pred_mae = float(np.mean(np.abs(pred_np[r, 0] - wet_np[r, 0])))
    ax.set_title(f"crop {r} - MAE {pred_mae:.4f} | ESR {float(esr_metric(tv, pv)):.4f}")
    ax.set_ylabel("amp"); ax.legend(loc="lower right", fontsize=8); ax.set_ylim(-1.05, 1.05)
axes[-1, 0].set_xlabel("Time (s)")
fig.suptitle(f"GR-TFiLM diffssl LSTM - best val loss {callbacks[0].best_model_score:.6f}", y=1.005)
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_output_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {plot_path}")
plt.show()


Loaded best checkpoint: /content/drive/Othercomputers/MacBook Air/data/diffssl_gr_tfilm_runs/gr_tfilm_20260708_100749_diffssl_lstm32_tvc_gr_tfilm/checkpoints/best-089-337500.ckpt


RuntimeError: cuDNN error: CUDNN_STATUS_NOT_SUPPORTED. This error may appear if you passed in a non-contiguous input.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"


: 